# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [15]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [16]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [17]:
print("Revenue before:", (df['qty'] * df['price'].str.replace('$', '', regex=False).astype(float)).sum())

Revenue before: 4852.5


In [18]:
before = len(df)

df = df.drop_duplicates()

log('duplicates', 'dropped exact duplicate rows', before - len(df))
print(before)


[duplicates] dropped exact duplicate rows (15 row(s))
315


In [19]:
print("Revenue after:", (df['qty'] * df['price'].str.replace('$', '', regex=False).astype(float)).sum())

Revenue after: 4594.5


### TODO 2 — clean `price` -> float

In [20]:
df['price'] = (

    df['price']

    .astype(str)

    .str.replace('$', '', regex=False)

    .str.strip()

    .astype(float)

)

log('price', 'removed dollar signs and converted price to float', 0)

[price] removed dollar signs and converted price to float (0 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [21]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

missing = df['qty'].isna().sum()

df = df.dropna(subset=['qty'])

log('missing_qty', 'dropped rows with missing quantity', missing)

negative = (df['qty'] < 0).sum()

df = df[df['qty'] >= 1]

log('negative_qty', 'dropped rows with negative quantity', negative)



[missing_qty] dropped rows with missing quantity (12 row(s))
[negative_qty] dropped rows with negative quantity (13 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [22]:
print(df['item'].value_counts())

ITEM_MAP = {

    'Cheeseburger': 'Cheeseburger',

    'cheese burger': 'Cheeseburger',

    'Foam Finger': 'Foam Finger',

    'foam finger': 'Foam Finger',

    'Rain Poncho': 'Rain Poncho',

    'rain poncho': 'Rain Poncho'

}

before_items = df['item'].copy()

df['item'] = df['item'].map(ITEM_MAP)

item_changed = (before_items != df['item']).sum()

log('item', 'canonicalized six spellings into three product names', item_changed)



item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] canonicalized six spellings into three product names (126 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [23]:
before_categories = df['category'].copy()

df['category'] = df['category'].replace({

    'food': 'Food',

    'rain-gear': 'RainGear',

    'Apparel': 'Merch'

})

category_changed = (before_categories != df['category']).sum()

log(

    'category',

    'normalized capitalization and combined Apparel with Merch',

    category_changed

)

[category] normalized capitalization and combined Apparel with Merch (132 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [24]:
assert df.duplicated().sum() == 0

assert df['qty'].min() >= 1

assert df['price'].dtype == float

assert set(df['item'].unique()) == {

    'Cheeseburger',

    'Foam Finger',

    'Rain Poncho'

}

assert set(df['category'].unique()) == {

    'Food',

    'Merch',

    'RainGear'

}

print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [25]:
df['revenue'] = df['qty'] * df['price']

revenue_by_category = (

    df.groupby('category')['revenue']

    .sum()

    .sort_values(ascending=False)

    .round(2)

)

print(revenue_by_category)

print('Overall total:', round(df['revenue'].sum(), 2))

category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Overall total: 4740.0


**What I would tell the vendor:** _..._

### TODO 8 — read back your log

In [26]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped exact duplicate rows,15
1,price,removed dollar signs and converted price to float,0
2,missing_qty,dropped rows with missing quantity,12
3,negative_qty,dropped rows with negative quantity,13
4,item,canonicalized six spellings into three product...,126
5,category,normalized capitalization and combined Apparel...,132


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

A)the biggest effect was dropping the duplicates. The numbers moved from Revenue before: 4852.5 to Revenue after: 4594.5. this is a difference of 258.

B) One decision that could've been done differently is using the negative quanities as refunds instead of dropping them. This lab said to drop the negative quantities but it could make sense to keep them since they could affect profit totals.